# 🦠 Thorough Master Notebook — Modeling Epidemic Outbreaks

*The project lead's verified source: both groups' methods in full, the
planning sections participants never see, and the deeper mathematics.
Participants work in `Group_A/` (equations) or `Group_B/` (coin flips);
they can read this notebook as an optional deep dive.*

### How to read this notebook

| Marker | What to do |
|---|---|
| 📖 **IDEA** | Read the explanation first |
| ✏️ **EDIT ME** | Change a value, re-run, and see what moves |
| ▶️ **RUN** | Run the code cell |
| 👀 **READ** | Inspect the result, graph, or message |
| 🧠 **BUILD IT** | A core concept turned into code — read this one closely |
| ✅ **CHECKPOINT** | Pause and answer the questions |
| 🔒 **RUN ONLY** | Infrastructure. Run it; you do not need to memorize it |
| ⭐ **OPTIONAL** | Try only after the required work is complete |

> 🖥️ **Before anything else — pick the kernel.** In the top-right corner of
> Jupyter, the kernel should read **`Python (epidemic-modeling)`**. Your site
> may name it differently — for example `Python (<site_kernel_name>)` — so if
> you don't see it, ask your project lead *before* debugging anything.
> A wrong kernel makes the setup cell fail with `ModuleNotFoundError`.

## How to use this master

Built and verified **first**, then copied and simplified into the group
notebooks (never the other way around). Everything runs top-to-bottom in
under a minute on a laptop — no GPU, no internet, nothing beyond
NumPy + matplotlib.

## 🧭 Python survival guide — read this once, then come back when needed

This project is for people from **all STEM backgrounds**. You do **not** need
to memorize Python syntax. When you see unfamiliar code, first identify the
job it is doing.

| Python word | Plain-English meaning | Example from this project |
|---|---|---|
| **variable** | A name that stores a value | `GAMMA = 0.25` |
| **function** | A reusable mini-program that performs one job | `simulate_sir(2.5)` |
| **argument** | A value you give to a function | `vaccinated_frac=0.6` |
| **return** | The value a function gives back | `return new_cases` |
| **list** | An ordered collection | `[0.0, 0.2, 0.4, 0.6]` |
| **NumPy array** | A grid of numbers supporting math on all values at once | `np.cumsum(observed)` |
| **for loop** | “Do this once per item” | `for v in VACCINATION_LIST:` |
| **if / break** | “When this is true, act / stop” | `if I == 0: break` |
| **random generator** | The coin we flip | `rng.binomial(S, p)` |
| **f-string** | A print message with values filled in | `print(f"R0 = {best_R0}")` |

### How to read a function

```python
def simulate_sir(R0, vaccinated_frac=0.0):   # name + inputs
    ...                                       # the daily loop
    return new_cases                          # give the curve back
```

Read that as: *“given an R₀ and a vaccination level, play the epidemic out
day by day, and hand back how many people got sick each day.”*

Whenever a cell is marked **🔒**, focus on the explanation above it — the 🧩
function map below tells you each tool's job.

## 1. Mentor alignment

**Research question (mentor-approved):** when does an outbreak explode —
and how much vaccination stops it?

**Learner outcome:** by the end, participants can explain what R₀ is,
measure it from a real-looking case curve, use a simulation model to test
vaccination scenarios, interpret the herd-immunity threshold 1 − 1/R₀, and
state one limitation (perfect mixing — no households or superspreaders).

**Result every participant must produce:** the vaccination-cliff figure
with their fitted R₀ and the formula's prediction marked.

**Misunderstanding to prevent:** “R₀ > 1 means the outbreak *always*
happens.” (Group B's extinction experiment kills this one with data.)

## 2. Time budget

In [ ]:
TIME_BUDGET_HOURS = {
    "setup_and_preflight":            0.5,
    "core_notebook_path":             3.5,
    "guided_experiments":             3.0,
    "optional_extensions":            0.5,
    "presentation_prep_and_practice": 2.5,
}
total = sum(TIME_BUDGET_HOURS.values())
for k, v in TIME_BUDGET_HOURS.items():
    print(f"  {k:32s} {v:4.1f} h")
print(f"  {'TOTAL':32s} {total:4.1f} h  (design limit: 10 h)")
assert total <= 10.0, "over the design limit — cut scope, not sleep"

## 3. Scientific story and concept map

> **When does an outbreak explode — and how much vaccination stops it?**

A disease has swept through a town of **10,000 people**. All we have is the
public-health record: how many people got sick each day, for 150 days
(`data/observed_outbreak.csv`). You are the disease detectives.

Epidemiologists describe outbreaks with three groups of people and one number:

| Symbol | Who they are |
|---|---|
| **S** — Susceptible | could still catch it |
| **I** — Infectious | sick now, and spreading it |
| **R** — Recovered | had it, now immune |

**R₀ ("R-naught")** = how many people one sick person infects, on average,
when everyone around them is susceptible. If R₀ > 1 the outbreak grows; if
R₀ < 1 it dies out. Your two jobs:

1. **Measure this outbreak's R₀** from the daily-case record.
2. **Find the vaccination level that would have prevented it** — and compare
   your answer with the famous formula for herd immunity, **v\* = 1 − 1/R₀**.

**Story map:** question (what stops an outbreak?) → evidence (150 days
of case counts) → method (fit a simulation model's R₀ to the curve) →
result (the vaccination cliff at 1 − 1/R₀) → meaning (herd immunity is a
threshold, not a slope — and near it, luck decides).

## 4. Environment and kernel

Pure NumPy + matplotlib — runs anywhere. Two reproducible specs ship with
the repo (`environment.yml`, `requirements.txt`); the README has the
conda/venv + kernel-registration commands.

### 🧩 Function map — everything `epidemic_helpers.py` gives you

You never need to read the helper module to do this project — this table is
its contract:

| Tool | Plain-English job |
|---|---|
| `simulate_sir(R0, ...)` | The equation model: plays the epidemic out day by day, returns the daily-cases curve. |
| `simulate_stochastic(R0, ...)` | The coin-flip model: one random epidemic (different every run). |
| `run_many(R0, n_runs)` | Repeats the random epidemic many times; returns all curves + final sizes. |
| `align_to_threshold(curve, 20)` | Shifts a curve so day 0 = the day it reached 20 total cases — makes epidemics comparable. |
| `rmse(a, b)` | The typical difference between two curves — our “how wrong is the model?” score. |
| `growth_rate(curve)` | The early exponential growth rate (the classic field shortcut — see §9 for its bias). |
| `analytic_final_size(R0)` | 100-year-old mathematics: the fraction ultimately infected, from R₀ alone. |

### What the next cell does 📖

1. **Imports the toolbox.** Think of an import as: *“Python, please give me
   this toolbox.”* — `numpy` does math on whole lists of numbers at once,
   `matplotlib` draws graphs.
2. **Loads the outbreak record** from `data/observed_outbreak.csv` into an
   array called `observed` — one number per day: how many people got sick.
3. **Loads three small helper tools** (explained right above their use) and
   **prints a check** so you know everything is ready.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt

# three small tested tools (🔒 in epidemic_helpers.py — see the master's 🧩 map):
#   align_to_threshold : shift a curve so day 0 = the day it reached 20 cases
#   rmse               : the typical difference between two curves
#   analytic_final_size: textbook prediction of the outbreak's final size
from epidemic_helpers import align_to_threshold, rmse, analytic_final_size

observed = np.loadtxt("data/observed_outbreak.csv",
                      delimiter=",", skiprows=1)[:, 1]

N = 10_000          # people in the town
GAMMA = 0.25        # recovery rate: 1 / (average 4 days infectious)

print(f"days of records: {len(observed)}")
print(f"total people infected: {int(observed.sum()) + 5} of {N}")
print("Preflight OK — you are ready to start.")

## 5. Data provenance and responsible use

The outbreak is **synthetic**: `data/generate_data.py` runs one seeded
stochastic epidemic (hidden truth: R₀ = 2.5, four infectious days) in a
town of 10,000. Fully reproducible, license-free, no real patients — and
participants aren't told the true R₀, so “measure it” is a real task.
Real-world extensions: WHO measles line lists, Our World in Data COVID
curves. Handle real epidemic data with care: it is about people.

## 6. Inspect the data

### What the next cell does 📖

Two pictures of the epidemic: **left** — how many people got sick each day
(the famous “epidemic curve”); **right** — the running total. Steps: plot the
daily numbers, add them up with `np.cumsum`, plot the total, label everything.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
days = np.arange(len(observed))

axes[0].bar(days, observed, width=1.0, color="C3", alpha=0.8)
axes[0].set_xlabel("day"); axes[0].set_ylabel("new cases per day")
axes[0].set_title("The outbreak, day by day")

axes[1].plot(days, np.cumsum(observed), color="C3")
axes[1].set_xlabel("day"); axes[1].set_ylabel("total people infected")
axes[1].set_title("The running total")

for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 7. Method A (Group A): the SIR equations


Our model is two sentences of common sense, turned into arithmetic:

1. **Some susceptible people get infected today.** The more infectious people
   are around, the more likely each susceptible person is to meet one.
2. **Some infectious people recover today.** On average people are infectious
   for 4 days, so about a quarter (`GAMMA = 0.25`) recover each day.

### The Python, decoded 🧩

| Line | What it does in plain English |
|---|---|
| `def sir_day(...):` | Defines a mini-program: give it today's S, I, R — it returns tomorrow's. |
| `np.exp(-beta * I / N)` | The chance one susceptible person gets through the whole day *without* being infected. |
| `S * (1 - ...)` | So this many susceptible people, on average, DO get infected today. |
| `gamma * I` | This many infectious people recover today. |
| `for day in range(days):` | Repeat the daily update 150 times — that is the whole simulation. |

In [ ]:
def sir_day(S, I, R, beta, gamma, N):
    """One day of the epidemic: tomorrow's counts from today's counts."""
    new_infections = S * (1 - np.exp(-beta * I / N))   # who catches it today
    recoveries = gamma * I                              # who recovers today
    return (S - new_infections,
            I + new_infections - recoveries,
            R + recoveries,
            new_infections)

def simulate_sir(R0, vaccinated_frac=0.0, days=150, I0=5):
    """Run the model day by day; return the daily new-case curve."""
    beta = R0 * GAMMA                       # transmission rate from R0
    S = N * (1 - vaccinated_frac) - I0      # everyone not vaccinated...
    I = I0                                  # ...except the first 5 cases
    R = N * vaccinated_frac                 # vaccinated start immune
    new_cases = np.zeros(days)
    for day in range(days):
        S, I, R, new_cases[day] = sir_day(S, I, R, beta, GAMMA, N)
    return new_cases

print("SIR model defined — two functions, no magic")

### What the next cell does 📖

A first honest guess: **R₀ = 2.0**. Steps:

1. simulate an epidemic with `simulate_sir(2.0)`,
2. align both curves at the day they reached 20 total cases (🔒
   `align_to_threshold` — random outbreaks take off at random times, so we
   compare them from the same milestone),
3. draw the model on top of reality.

**Predict before you run:** will R₀ = 2.0 be too small, too big, or just right?

In [ ]:
obs_aligned = align_to_threshold(observed, 20)

guess = simulate_sir(2.0)
guess_aligned = align_to_threshold(guess, 20)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.5, label="observed outbreak")
ax.plot(guess_aligned, color="C0", lw=2, label="SIR model, R0 = 2.0")
ax.set_xlabel("days since the outbreak reached 20 cases")
ax.set_ylabel("new cases per day")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"difference between model and reality: {rmse(obs_aligned, guess_aligned):.1f} cases/day")

## 🧪 Measure R₀ — try every candidate, keep the best

This is how a lot of real science works: propose candidate values, simulate
each one, and keep the candidate whose prediction best matches reality.

### What the next cell does 📖

1. loops over a list of candidate R₀ values,
2. simulates one epidemic per candidate and measures its difference from the
   observed curve (`rmse` — lower is better),
3. plots difference vs candidate — the dip marks your measurement,
4. redraws the best model on top of reality.

In [ ]:
R0_CANDIDATES = [1.5, 1.8, 2.0, 2.2, 2.4, 2.5, 2.6, 2.8, 3.0, 3.5]
VACCINATION_LIST = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.55, 0.6, 0.65, 0.7, 0.8]

errors = []
for R0 in R0_CANDIDATES:
    model = align_to_threshold(simulate_sir(R0), 20)
    errors.append(rmse(obs_aligned, model))

best_R0 = R0_CANDIDATES[int(np.argmin(errors))]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(R0_CANDIDATES, errors, "o-")
axes[0].axvline(best_R0, color="C3", ls=":")
axes[0].set_xlabel("candidate R0"); axes[0].set_ylabel("difference from reality")
axes[0].set_title(f"The dip is the answer: R0 = {best_R0}")

best = align_to_threshold(simulate_sir(best_R0), 20)
axes[1].bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
            alpha=0.5, label="observed")
axes[1].plot(best, color="C0", lw=2, label=f"SIR, R0 = {best_R0}")
axes[1].set_xlabel("days since 20 cases"); axes[1].legend()
axes[1].set_title("Best-fitting epidemic")
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"Your measurement: this outbreak's R0 ≈ {best_R0}")
print(f"Herd-immunity formula predicts the threshold: 1 - 1/R0 = {1 - 1/best_R0:.2f}")

## 🧪 The shared experiment — how much vaccination stops it?

Vaccinating a fraction `v` of the town moves them straight from S to R on
day 0: they can neither catch nor spread it. Somewhere between v = 0 and
v = 0.8 the outbreak should collapse — that point is the **herd-immunity
threshold**, and the textbook formula says it sits at **1 − 1/R₀**.

### What the next cell does 📖

1. loops over your `VACCINATION_LIST`,
2. simulates one epidemic per vaccination level (your fitted R₀),
3. plots the outbreak's final size against vaccination, with the formula's
   prediction as a vertical dashed line.

**Predict before you run:** will the curve slope down gently, or fall off a
cliff?

In [ ]:
final_sizes = []
for v in VACCINATION_LIST:
    cases = simulate_sir(best_R0, vaccinated_frac=v)
    final_sizes.append(cases.sum())

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(VACCINATION_LIST, final_sizes, "o-", color="C2")
ax.axvline(1 - 1/best_R0, color="k", ls="--",
           label=f"herd-immunity formula: 1 - 1/R0 = {1 - 1/best_R0:.2f}")
ax.set_xlabel("fraction of the town vaccinated on day 0")
ax.set_ylabel("total people infected")
ax.set_title("The vaccination cliff")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# 8. Method B (Group B): the coin-flip model


Group A's equations track *average* behavior — 12.7 infections per day is a
fine average, but no real town infects 0.7 of a person. Your model tracks
**whole people and luck**: each day, every susceptible person either escapes
infection or doesn't — decided by a (weighted) coin flip.

### The Python, decoded 🧩

| Line | What it does in plain English |
|---|---|
| `rng = np.random.default_rng()` | A random-number generator — the coin we flip. |
| `1 - np.exp(-beta * I / N)` | Today's chance that one susceptible person gets infected. |
| `rng.binomial(S, p)` | Flip that weighted coin for **all S people at once**; count the infections. |
| `rng.binomial(I, GAMMA)` | Each sick person recovers today with chance `GAMMA`; count who does. |
| `if I == 0: break` | No one left infectious → the outbreak is over, stop early. |

In [ ]:
def simulate_outbreak(R0, vaccinated_frac=0.0, days=150, I0=5, rng=None):
    """One RANDOM epidemic — a different story every time you run it."""
    if rng is None:
        rng = np.random.default_rng()       # unseeded = truly random
    beta = R0 * GAMMA
    S = int(N * (1 - vaccinated_frac)) - I0
    I = I0
    new_cases = np.zeros(days)
    for day in range(days):
        p = 1 - np.exp(-beta * I / N)        # today's infection risk
        infections = rng.binomial(S, p)      # coin flips for all S people
        recoveries = rng.binomial(I, GAMMA)  # coin flips for all I people
        S -= infections
        I += infections - recoveries
        new_cases[day] = infections
        if I == 0:
            break                            # outbreak over
    return new_cases

print("stochastic model defined — run the NEXT cell twice and compare!")

### What the next cell does 📖

Runs **one** random epidemic at R₀ = 2.5 and plots it against the observed
outbreak.

**Now the important part: run this cell two or three times** (click it, press
Shift+Enter, repeat). Same disease, same town, same R₀ — different epidemic
every time. *That* is what the equations can't show you.

In [ ]:
one_run = align_to_threshold(simulate_outbreak(2.5), 20)
obs_aligned = align_to_threshold(observed, 20)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.5, label="observed outbreak")
ax.plot(one_run, color="C0", lw=2, label="one random epidemic (R0 = 2.5)")
ax.set_xlabel("days since 20 cases"); ax.set_ylabel("new cases per day")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Re-run me! I am different every time")
plt.tight_layout(); plt.show()

## 👀 One run means nothing — run two hundred

With randomness, a single simulation is an anecdote. The honest move is to
run **many** and look at the whole cloud.

### What the next cell does 📖

1. runs 200 random epidemics at R₀ = 2.5 (a small loop),
2. draws every one as a faint line — a “spaghetti plot”,
3. lays the observed outbreak on top.

**Predict before you run:** will reality sit inside the spaghetti, or outside it?

In [ ]:
N_SIMULATIONS = 200

rng = np.random.default_rng(0)          # seeded, so everyone gets the same cloud

fig, ax = plt.subplots(figsize=(8.5, 4))
final_sizes = []
for i in range(N_SIMULATIONS):
    run = simulate_outbreak(2.5, rng=rng)
    final_sizes.append(run.sum())
    aligned = align_to_threshold(run, 20)
    if len(aligned) > 0:
        ax.plot(aligned, color="C0", alpha=0.08)
ax.bar(range(len(obs_aligned)), obs_aligned, width=1.0, color="C3",
       alpha=0.55, label="observed outbreak")
ax.set_xlabel("days since 20 cases"); ax.set_ylabel("new cases per day")
ax.set_title(f"{N_SIMULATIONS} possible epidemics, one reality")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"epidemics that fizzled early (< 500 total cases): "
      f"{int(np.sum(np.array(final_sizes) < 500))} of {N_SIMULATIONS}")

## 🧪 The luck experiment — one sick traveler arrives

Equations say: R₀ = 2.5 > 1, so an outbreak *must* grow. Coin flips disagree:
if the **first** sick person happens to recover before infecting anyone, the
outbreak dies at one case. Theory says that lucky escape happens with
probability about **1/R₀ = 0.4**.

### What the next cell does 📖

1. runs 500 epidemics that each start from **a single case** (`I0=1`),
2. counts how many fizzle out (fewer than 500 total cases),
3. compares that fraction with the 1/R₀ prediction.

In [ ]:
rng = np.random.default_rng(4)
fizzled = 0
for _ in range(500):
    run = simulate_outbreak(2.5, I0=1, rng=rng)
    if run.sum() < 500:
        fizzled += 1

print(f"outbreaks that fizzled: {fizzled} of 500  ({fizzled/500:.0%})")
print(f"theory's prediction:    about 1/R0 = {1/2.5:.0%}")
print("\nSame town, same disease — sometimes the whole outbreak")
print("comes down to a few coin flips at the start.")

## 🧪 The shared experiment — how much vaccination stops it?

Same design as Group A, but with randomness the honest question changes from
*“how big is the outbreak?”* to *“what is the **probability** of an outbreak?”*

### What the next cell does 📖

1. loops over your `VACCINATION_LIST`,
2. runs 100 random epidemics per vaccination level (your fitted R₀),
3. plots the fraction that became real outbreaks (> 500 cases), with the
   herd-immunity formula as a dashed line.

**Predict before you run:** at exactly the threshold, is the outbreak
probability 0, 1, or something in between?

In [ ]:
rng = np.random.default_rng(3)
outbreak_prob = []
for v in VACCINATION_LIST:
    outbreaks = 0
    for _ in range(100):
        run = simulate_outbreak(best_R0, vaccinated_frac=v, rng=rng)
        if run.sum() > 500:
            outbreaks += 1
    outbreak_prob.append(outbreaks / 100)

fig, ax = plt.subplots(figsize=(8, 3.8))
ax.plot(VACCINATION_LIST, outbreak_prob, "o-", color="C2")
ax.axvline(1 - 1/best_R0, color="k", ls="--",
           label=f"herd-immunity formula: 1 - 1/R0 = {1 - 1/best_R0:.2f}")
ax.set_xlabel("fraction of the town vaccinated on day 0")
ax.set_ylabel("probability of a real outbreak")
ax.set_title("The vaccination cliff — probabilistic edition")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. Master-only depth: three bridges to the textbooks 📚

**1. The final-size equation.** A century-old result: the fraction Z of
the population ultimately infected solves `Z = 1 − exp(−R₀·Z)` — no
simulation needed. The next cell checks our models against it.

**2. The field shortcut and its bias.** Epidemiologists often estimate R₀
from the early growth rate r as R₀ ≈ 1 + r/γ. On this outbreak it gives
≈ 2.1 — biased low, because by the time the curve is measurable the town
is already running out of susceptibles. A good talking point: *every
method has assumptions; know when yours bend.*

**3. Deterministic vs stochastic.** Same mechanics, two philosophies: the
equations are the *average* of infinitely many coin-flip epidemics — but
averages hide extinction and luck. That is exactly the groups' comparison.

### What the next cell does 📖

1. solves the final-size equation for the fitted R₀ (🔒 helper),
2. compares three answers for “how many people get infected”: observed,
   the SIR model, and the 100-year-old formula,
3. computes the field-shortcut R₀ estimate and shows its low bias.

In [ ]:
Z = analytic_final_size(best_R0)
sir_total = simulate_sir(best_R0).sum()
print(f"{'':28s}{'total infected':>16s}")
print(f"{'observed outbreak':28s}{int(observed.sum()) + 5:>16d}")
print(f"{'SIR model (fitted R0)':28s}{sir_total:>16.0f}")
print(f"{'final-size equation':28s}{Z * N:>16.0f}")

from epidemic_helpers import growth_rate
r = growth_rate(obs_aligned, 0, 15)
print(f"\nfield shortcut: R0 ≈ 1 + r/GAMMA = {1 + r/GAMMA:.2f} "
      f"(vs fitted {best_R0}) — biased low; see the note above")

## 10. HPC execution

*Deliberately minimal:* every cell runs in seconds. But Group B's design —
hundreds of independent random simulations — is **embarrassingly
parallel**, the exact shape of an HPC array job (each task runs a slice of
simulations; a merge step combines them). Real epidemic models
(agent-based, city-scale) run exactly this way on clusters. Site
placeholders for the README: `<python_or_conda_module>`,
`<site_approved_environment_directory>`, `<participant_work_directory>`.
**System-change rule:** confirm current site names at the bootcamp —
never assume last year's.

## 11. Group differentiation plan

Split **by modeling philosophy**; identical data, question, R₀
measurement task, and vaccination experiment.

| | Group A | Group B |
|---|---|---|
| Model | SIR equations (deterministic) | coin-flip simulation (stochastic) |
| Builds | `sir_day` + `simulate_sir` (~20 lines) | `simulate_outbreak` (~18 lines) |
| Fits R₀ by | best single curve | best *median of 100* curves |
| Special insight | sharp cliff at 1 − 1/R₀ | soft cliff + stochastic extinction |
| Expected R₀ | ≈ 2.4 | ≈ 2.5 |
| Runtime | seconds | ~10–30 s per sweep |

## 12. 📊 Variable and column reference

Use this table to make **your own extra graphs** without guessing what
names mean. Everything listed is in memory after running the notebook.

| Name | What it is | Good for plotting |
|---|---|---|
| `observed` | array, 150 days of new cases | the epidemic curve |
| `obs_aligned` | the same curve, day 0 = 20th case | comparisons with models |
| `N`, `GAMMA` | town size (10,000) and recovery rate (0.25/day) | — |
| `best_R0` | your measurement of this outbreak's R₀ | headline number |
| `R0_CANDIDATES`, `VACCINATION_LIST` | your ✏️ experiment settings | axes |
| `errors` | difference-from-reality per candidate | the fitting dip |
| `simulate_sir` / `simulate_outbreak` | both models (defined above) | any what-if |
| `final_sizes`, `outbreak_prob` | the two vaccination sweeps | both cliffs |
| `analytic_final_size(R0)` | the textbook check | theory vs simulation |

*Example:* `plt.plot(np.cumsum(observed))` — the outbreak's running total.

## 13. Presentation checkpoint

Every team explains: the question (why outbreaks stop), the evidence (the
case curve), the method (fit R₀ by simulation), the key figure (the
vaccination cliff with 1 − 1/R₀ marked), one experiment of their own, and
one limitation + next step (networks, households, behavior change).
Budget 1.5–3 h of the 10–12 h total. The joint moment: **equations and
coin flips independently agree the threshold is ≈ 60%.**

## 14. Optional deeper material ⭐

1. **SEIR:** add an Exposed (infected-but-not-yet-infectious) stage — one
   more compartment, one more line in `sir_day`.
2. **Behavior change:** cut beta in half when daily cases pass 200
   (people get cautious). What happens to the peak? The final size?
3. **Waves:** let immunity fade (R → S at 1%/day). Can you make a second
   wave?
4. **The (1/R₀)^I₀ law:** verify the extinction probability formula
   against simulations for I₀ = 1..5.

## 15. Project lead release checklist

- [x] Mentor approved the research question and the learner outcome
- [x] Mentor approved how Group A and Group B differ (by modeling philosophy)
- [x] Every notebook runs top-to-bottom in a fresh session
- [x] Group paths tested independently; runtimes fit the 10–12 h budget
- [x] Data ships with the repo + seeded regeneration script for provenance
- [x] Environment reproducible two ways (`environment.yml`, `requirements.txt`)
- [x] Every code cell has a “What the next cell does” explanation
- [x] No credentials, no personal data, no oversized files
- [ ] Fresh-clone test on the *actual host site* by a peer mentor
- [ ] Program organizer approval to add to the official bootcamp GitHub